In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import spacy

# doc2vecを使うためのライブラリ
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv


In [2]:
# データの読み込み
train_df = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv')
test_df = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv')

submission_df = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv')


# データの確認
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17307 entries, 0 to 17306
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   essay_id   17307 non-null  object
 1   full_text  17307 non-null  object
 2   score      17307 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 405.8+ KB


In [3]:
train_df.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [4]:
train_df.head(2).full_text.values

array(['Many people have car where they live. The thing they don\'t know is that when you use a car alot of thing can happen\xa0like you can get in accidet or\xa0the smoke that the car has is bad to breath\xa0on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban\'s families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden\xa0on the outskirts of freiburd that near the French and Swiss borders. You probaly won\'t see a car in Vauban\'s streets because they are completely "car free" but\xa0If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend in Europe,The untile st

In [5]:
train_df.head(2).score.values

array([3, 3])

In [6]:
#シンプルにDec2Vecを使ってみる
# データの前処理
nlp = spacy.load('en_core_web_sm')

# テキストの前処理
def preprocess_text(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop]


In [7]:
# テキストの前処理

train_df['full_text_'] = train_df['full_text'].apply(preprocess_text)

In [8]:
train_df.tail(2)

,essay_id,full_text,score,full_text_
17305,fffb49b,"In ""The Challenge of Exporing Venus,"" the auth...",1,"["", challenge, Exporing, Venus, ,, "", author, ..."
17306,fffed3e,Venus is worthy place to study but dangerous. ...,2,"[Venus, worthy, place, study, dangerous, ., re..."


In [9]:
full_text_series = train_df['full_text_']

full_text_series[:2]

0    [people, car, live, ., thing, know, use, car, ...
1    [scientist, NASA, discuss, ", face, ", mar, .,...
Name: full_text_, dtype: object

In [10]:
tagged_data = [TaggedDocument(words=doc, tags=[i]) for i, doc in enumerate(full_text_series)]

In [11]:
len(tagged_data) == len(train_df)

True

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [13]:
# モデルの学習
model = Doc2Vec(vector_size=300, window=5, min_count=1, workers=4)


In [14]:

model.build_vocab(tagged_data)


In [15]:

model.train(tagged_data, total_examples=model.corpus_count, epochs=100)

In [16]:
# ベクトルの取得

vectors_ = [model.infer_vector(doc) for doc in full_text_series]

vectors_[:2]

[array([-3.67572188e-01, -5.81442833e-01, -5.37984431e-01, -1.97785354e+00,
        -2.58478075e-01,  1.08138120e+00,  1.71891725e+00, -8.94088328e-01,
        -1.25482416e+00, -1.23267233e+00,  1.31274194e-01,  1.82513773e-01,
        -1.55877090e+00,  1.09710775e-01, -3.78429517e-02, -1.00583065e+00,
         8.31118822e-02, -7.92860091e-01, -3.87062758e-01, -1.39483488e+00,
        -9.07627225e-01, -7.44669497e-01,  2.87938744e-01, -9.77331579e-01,
        -3.30037981e-01, -6.67116344e-01, -1.76833856e+00,  1.41257262e+00,
         1.53115427e+00,  7.81967282e-01, -3.49227399e-01, -2.31198460e-01,
         4.66660380e-01, -7.93569386e-01,  9.94111225e-02, -1.12662446e+00,
         2.26989341e+00,  5.42326212e-01,  1.47476411e+00, -1.13304520e+00,
         3.68145376e-01,  6.58291280e-01, -2.69858146e+00, -2.16137301e-02,
        -5.93902409e-01, -3.07373941e-01,  1.33315277e+00, -6.29561186e-01,
         5.58357894e-01,  6.25317752e-01,  7.65040755e-01, -7.17988133e-01,
         1.4

In [17]:

vectors = np.array(vectors_)
vectors[:2]

array([[-3.67572188e-01, -5.81442833e-01, -5.37984431e-01,
        -1.97785354e+00, -2.58478075e-01,  1.08138120e+00,
         1.71891725e+00, -8.94088328e-01, -1.25482416e+00,
        -1.23267233e+00,  1.31274194e-01,  1.82513773e-01,
        -1.55877090e+00,  1.09710775e-01, -3.78429517e-02,
        -1.00583065e+00,  8.31118822e-02, -7.92860091e-01,
        -3.87062758e-01, -1.39483488e+00, -9.07627225e-01,
        -7.44669497e-01,  2.87938744e-01, -9.77331579e-01,
        -3.30037981e-01, -6.67116344e-01, -1.76833856e+00,
         1.41257262e+00,  1.53115427e+00,  7.81967282e-01,
        -3.49227399e-01, -2.31198460e-01,  4.66660380e-01,
        -7.93569386e-01,  9.94111225e-02, -1.12662446e+00,
         2.26989341e+00,  5.42326212e-01,  1.47476411e+00,
        -1.13304520e+00,  3.68145376e-01,  6.58291280e-01,
        -2.69858146e+00, -2.16137301e-02, -5.93902409e-01,
        -3.07373941e-01,  1.33315277e+00, -6.29561186e-01,
         5.58357894e-01,  6.25317752e-01,  7.65040755e-0

In [18]:

vectors.shape

(17307, 300)

In [19]:
# データの分割

X = vectors

y = train_df['score']


In [20]:
# yの値が１～６なので、LihgtGBMのクラスに合わせて、０～５に並べ替えるために書き換える

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_ = le.fit_transform(y)

In [21]:
print(y_.min())
print(y_.max())

0
5


In [22]:
print(X.shape)
print(y_.shape)

(17307, 300)
(17307,)


In [23]:
y_.dtype

dtype('int64')

In [24]:

X_train, X_valid, y_train, y_valid = train_test_split(X, y_, test_size=0.2, random_state=0)

# y_　は０～５の６クラス

In [25]:
# モデルの学習

lgb_train = lgb.Dataset(X_train, y_train)

lgb_valid = lgb.Dataset(X_valid, y_valid, reference=lgb_train)

params = {
    'objective': 'multiclass', # 多クラス分類
    'num_class': 6, # クラスの数
    'metric': 'multi_logloss' # 損失関数にmulti_loglossを使用
}

lgb_model = lgb.train(params, lgb_train, valid_sets=lgb_valid, num_boost_round=1000,
                     #  early_stopping_rounds=10
                     )


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062043 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 13845, number of used features: 300
[LightGBM] [Info] Start training from score -2.619956
[LightGBM] [Info] Start training from score -1.300584
[LightGBM] [Info] Start training from score -1.011908
[LightGBM] [Info] Start training from score -1.477036
[LightGBM] [Info] Start training from score -2.911614
[LightGBM] [Info] Start training from score -4.756556
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

In [26]:
# モデルの評価

y_pred = lgb_model.predict(X_valid)

y_pred


array([[2.52988300e-04, 5.44516252e-01, 4.54062233e-01, 1.16852380e-03,
        6.83443409e-11, 2.66777842e-09],
       [4.19151742e-06, 9.89917476e-01, 1.00543889e-02, 2.39423480e-05,
        7.18783240e-14, 8.47005872e-10],
       [1.27747488e-05, 6.14014413e-02, 8.34814240e-01, 1.03770999e-01,
        4.57595095e-07, 8.72131836e-08],
       ...,
       [2.26518580e-06, 5.14822627e-04, 9.87161363e-01, 1.23061139e-02,
        8.96096262e-06, 6.47385234e-06],
       [4.47718696e-04, 3.94147004e-01, 6.00498873e-01, 4.90630987e-03,
        3.25409129e-08, 6.27079446e-08],
       [8.75687148e-02, 8.92558106e-01, 1.98723385e-02, 8.40412781e-07,
        5.31912511e-14, 7.10106233e-11]])

In [27]:
# 予測値をクラスに変換
y_pred = np.argmax(y_pred, axis=1)

mean_squared_error(y_valid, y_pred)

0.6285384170999422

In [28]:
test_df['full_text'] = test_df['full_text'].apply(preprocess_text)

In [29]:
test_full_text_series = test_df['full_text']

test_full_text_series[:3]

0    [people, car, live, ., thing, know, use, car, ...
1    [scientist, NASA, discuss, ", face, ", mar, .,...
2    [People, wish, technology, see, movie, ,, good...
Name: full_text, dtype: object

In [30]:
tagged_data_test = [TaggedDocument(words=doc, tags=[i]) for i, doc in enumerate(test_full_text_series)]

In [31]:
model.train(tagged_data_test, total_examples=model.corpus_count, epochs=100)

In [32]:
# ベクトルの取得

vectors_test_ = [model.infer_vector(doc) for doc in test_full_text_series]


In [33]:
vectors_test = np.array(vectors_test_)

In [34]:
vectors_test.shape

(3, 300)

In [35]:
prediction = lgb_model.predict(vectors_test)

In [36]:
print(prediction.max(), prediction.min())

0.9997450998052889 8.197388488064299e-20


In [37]:
# inversする
predict = np.argmax(prediction, axis=1)
predict

array([0, 1, 1])

In [38]:
# ラベルをinversする
Y_pre =le.inverse_transform(predict)
Y_pre


array([1, 2, 2])

In [39]:
submission_df['score'] =Y_pre
submission_df

,essay_id,score
0,000d118,1
1,000fe60,2
2,001ab80,2


In [40]:
submission_df.to_csv('/kaggle/working/submission.csv', index=False)